# 1. Import libaries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 2. Rescaling

In [8]:
# Load rfm data
raw_rfm = pd.read_parquet("data/staging/raw_rfm.parquet", engine= "pyarrow")
raw_rfm.head()

,customer_id,recency,frequency,monetary,loyalty_points,rating,quantity,gender,age_group,city,is_churned
0,C5663,35,38,7391.35,238,3,5,Male,Adult,Peshawar,0
1,C2831,271,24,2868.12,81,2,3,Male,Adult,Multan,0
2,C2851,105,42,1765.02,82,3,2,Other,Senior,Multan,1
3,C1694,30,27,925.20,45,2,4,Female,Senior,Peshawar,1
4,C4339,245,35,1156.69,418,3,1,Other,Senior,Lahore,1


In [9]:
# Clone the RFM values to a new dataframe
rfm_df = raw_rfm[[ 'recency', 'frequency', 'monetary']].copy()

# Instantiate
scaler = StandardScaler()

# Fit_transform
rfm_df_scaled = scaler.fit_transform(rfm_df)
rfm_df_scaled

array([[-1.40561075,  0.8842157 ,  2.84984141],
       [ 0.825675  , -0.09086857,  0.27210362],
       [-0.7437887 ,  1.1628112 , -0.35654063],
       ...,
       [ 0.49476398,  0.39667356,  0.93043908],
       [ 0.52312778,  0.81456682,  1.43255664],
       [-1.06524513, -1.62314384,  0.57168843]], shape=(6000, 3))

In [10]:
# Convert to dataframe
rfm_df_scaled = pd.DataFrame(rfm_df_scaled)
rfm_df_scaled.columns = ['recency', 'frequency', 'monetary']
rfm_df_scaled.head()

,recency,frequency,monetary
0,-1.405611,0.884216,2.849841
1,0.825675,-0.090869,0.272104
2,-0.743789,1.162811,-0.356541
3,-1.452884,0.118078,-0.835145
4,0.579855,0.675269,-0.703221


## 3. Hopkin Statistics

In [11]:
# snipset of the Hopkins statistic function to find if the dataset is suitable for clustering or not
from sklearn.neighbors import NearestNeighbors
from random import sample
from numpy.random import uniform
import numpy as np
from math import isnan

def hopkins(X):
    d = X.shape[1]
    #d = len(vars) # columns
    n = len(X) # rows
    m = int(0.1 * n) 
    nbrs = NearestNeighbors(n_neighbors=1).fit(X.values)
 
    rand_X = sample(range(0, n, 1), m)
 
    ujd = []
    wjd = []
    for j in range(0, m):
        u_dist, _ = nbrs.kneighbors(uniform(np.amin(X,axis=0),np.amax(X,axis=0),d).reshape(1, -1), 2, return_distance=True)
        ujd.append(u_dist[0][1])
        w_dist, _ = nbrs.kneighbors(X.iloc[rand_X[j]].values.reshape(1, -1), 2, return_distance=True)
        wjd.append(w_dist[0][1])
 
    H = sum(ujd) / (sum(ujd) + sum(wjd))
    if isnan(H):
        print(ujd, wjd)
        H = 0
 
    return H

In [12]:
# Pass a dataframe to the Hopkins statistic
hopkins(rfm_df_scaled)

np.float64(0.6238758449955042)